# Run Infomap on GraphRAG-style tables

GraphRAG pipelines build a community hierarchy over an entity graph, usually with
Leiden. Infomap is a drop-in alternative that finds communities by compressing
flow rather than maximizing modularity. This notebook reads the GraphRAG
`entities.parquet` and `relationships.parquet` tables, runs Infomap, writes
GraphRAG-style community outputs, and compares the result with Leiden.

For a first Infomap run, see the {doc}`quickstart <quickstart>`; for the options
used here, see the {doc}`options guide <options-guide>`.

In [ ]:
from pathlib import Path
import tempfile

import pandas as pd

from infomap.graphrag import read_graphrag, run_graphrag_communities

try:
    import igraph as ig
except ImportError:
    ig = None

## Build a small entity graph

GraphRAG stores entities and their relationships as two Parquet tables. Here you
build a tiny example: two triangles, Alpha-Beta-Gamma and Delta-Epsilon-Zeta,
joined by one weak link, so the community structure is easy to check.

In [ ]:
work_dir = Path(tempfile.mkdtemp(prefix="infomap-graphrag-"))
input_dir = work_dir / "input"
output_dir = work_dir / "infomap"
input_dir.mkdir()

entities = pd.DataFrame(
    {
        "id": ["a", "b", "c", "d", "e", "f"],
        "title": ["Alpha", "Beta", "Gamma", "Delta", "Epsilon", "Zeta"],
    }
)
relationships = pd.DataFrame(
    {
        "id": ["ab", "bc", "ca", "de", "ef", "fd", "cd"],
        "source": ["Alpha", "Beta", "Gamma", "Delta", "Epsilon", "Zeta", "Gamma"],
        "target": ["Beta", "Gamma", "Alpha", "Epsilon", "Zeta", "Delta", "Delta"],
        "weight": [2.0, 2.0, 2.0, 3.0, 3.0, 3.0, 1.0],
    }
)

entities.to_parquet(input_dir / "entities.parquet")
relationships.to_parquet(input_dir / "relationships.parquet")

## Read the tables into Infomap

`read_graphrag` maps entity titles to node ids and loads the weighted edges. The
table below shows each relationship with the node ids Infomap will use.

In [ ]:
graph = read_graphrag(
    input_dir / "entities.parquet", input_dir / "relationships.parquet"
)

relationships.assign(source_node=graph.sources, target_node=graph.targets)

## Run Infomap

`run_graphrag_communities` clusters the graph and writes GraphRAG-style outputs to
`output_dir`. Set `seed` and `num_trials` for a reproducible, stable result, the
same way you would for any Infomap run.

In [ ]:
result = run_graphrag_communities(
    input_dir=input_dir,
    output_dir=output_dir,
    silent=True,
    seed=123,
    num_trials=5,
)

result.infomap.codelength

## Inspect the communities

Infomap returns the per-node assignments and a GraphRAG-style community table with
one row per community, its level in the hierarchy, and its size.

In [ ]:
infomap_nodes = result.nodes
infomap_nodes

In [ ]:
infomap_communities = result.communities
infomap_communities

## Compare with Leiden

Leiden is GraphRAG's default community detector. On this graph both methods recover
the two triangles. Infomap reports the result as a flow-based hierarchy, while
Leiden returns a flat modularity partition.

In [ ]:
def _top_level_groups(communities):
    return [
        sorted(entity_ids)
        for entity_ids in communities.loc[communities["level"] == 0, "entity_ids"]
    ]


comparison_rows = [
    {
        "method": "Infomap",
        "number of communities": len(infomap_communities),
        "levels": int(infomap_communities["level"].nunique()),
        "largest community size": int(infomap_communities["size"].max()),
        "entity groups at top level": _top_level_groups(infomap_communities),
    }
]

if ig is None:
    print("Install python-igraph to run the Leiden comparison.")
else:
    leiden_graph = ig.Graph.TupleList(
        relationships[["source", "target", "weight"]].itertuples(
            index=False, name=None
        ),
        directed=False,
        edge_attrs=["weight"],
    )
    leiden_partition = leiden_graph.community_leiden(
        weights="weight",
        objective_function="modularity",
    )
    leiden_groups = [
        sorted(leiden_graph.vs[vertex]["name"] for vertex in community)
        for community in leiden_partition
    ]
    comparison_rows.append(
        {
            "method": "Leiden",
            "number of communities": len(leiden_partition),
            "levels": 1,
            "largest community size": max(len(group) for group in leiden_groups),
            "entity groups at top level": leiden_groups,
        }
    )

pd.DataFrame(comparison_rows)

## Where to go next

- {doc}`quickstart <quickstart>` for a first end-to-end Infomap run.
- {doc}`options guide <options-guide>` for what `seed`, `num_trials`, and the other
  options do.
- {doc}`compare-infomap-louvain-leiden-igraph <compare-infomap-louvain-leiden-igraph>`
  for a closer Infomap, Louvain, and Leiden comparison.